# Split Creation Pipeline Report

## Overview

This script creates a comprehensive train/test split system for a bioacoustics dataset, implementing spatial and temporal hold-out strategies alongside intelligent training set selection. The pipeline ensures rigorous evaluation by preventing data leakage while maximizing training set diversity and species representation.

### Main Elements

**Input**: A canonical clip index (Parquet format) containing audio clip metadata, embeddings, species predictions, and file paths.

**Output**: An annotated index with split labels (`train`, `dev`, `test_spatial`, `test_temporal`, `pool`, `excluded`) and boolean flags for each split category.

**Core Strategy**:
1. **Spatial Test Split**: 350 clips from 5 held-out sites (70 clips/site)
2. **Temporal Test Split**: 350 clips from edge-of-month days (50 clips/month × 7 months)
3. **Diversity Training**: 1,400 clips selected via k-means medoids across month × session strata
4. **Species Enrichment**: 900 clips covering top 60 species using farthest-first traversal
5. **Dev Split**: 300 clips held out from training for calibration

**Key Constraints**:
- No duplicate minute-level files
- Minimum 2-minute spacing within site-day combinations
- Site caps to ensure geographic diversity
- No overlap between test and training sets

---

## Code Architecture

### 1. Configuration & Setup



In [ ]:
RANDOM_SEED = 42
N_SPATIAL_TEST_SITES = 5
CLIPS_PER_SPATIAL_SITE = 70
TEMPORAL_EDGE_DAYS_START = 3
MIN_SPACING_MINUTES = 2



**Purpose**: Defines all hyperparameters as module-level constants for easy tuning. Uses descriptive names that document the split design.

**Logging**: Configured to stream INFO-level messages to stdout with timestamps, enabling real-time progress monitoring during long-running operations.

---

### 2. Utility Functions

#### `set_seed(seed: int) -> np.random.Generator`
Creates a NumPy random number generator with a fixed seed, ensuring reproducibility across runs while avoiding global state pollution.

#### `get_base_site(site: str) -> str`
Normalizes site names by removing numeric suffixes (e.g., "A4-2" → "A4"). This groups physically co-located recorders that were deployed as pairs/replacements.

#### `get_minute_key(row: dict) -> str`
Generates a unique identifier for each minute-level audio file using `{site}_{file_name}`. Prevents selecting multiple clips from the same source recording.

#### `check_spacing_constraint(...) -> bool`
Validates that a candidate clip's start time is at least `MIN_SPACING_MINUTES` (120 seconds) away from all previously selected clips on the same site-day. Prevents temporal autocorrelation in the dataset.

#### `filter_with_constraints(...) -> pl.DataFrame`
**Core selection engine** that implements all spatial and temporal constraints:

1. **Input sanitization**: Creates local copies of tracking sets to avoid side effects
2. **Shuffling**: Randomizes clip order using the seeded RNG
3. **Greedy selection loop**:
   - Checks minute key uniqueness
   - Validates 2-minute spacing on same site-day
   - Enforces per-site caps if specified
   - Updates tracking structures upon acceptance
4. **Returns**: Polars DataFrame of selected clips (empty if constraints cannot be satisfied)

This function is reused across all split creation steps, ensuring consistent constraint enforcement.

---

### 3. Embedding-Based Selection

#### `load_embeddings_for_clips(...) -> Dict[str, np.ndarray]`
**Efficient embedding loader** with practical limitations:

- **File grouping**: Groups clips by embedding path to minimize I/O operations
- **Prioritization**: Processes files with the most clips first (sorts by count descending)
- **Sampling limits**: 
  - Max 50 files loaded
  - Max 500 clips per file
- **Column detection**: Identifies embedding columns by prefix ('emb') or float dtype
- **Chunk index lookup**: Uses dictionary for O(1) lookups instead of linear search
- **Error handling**: Logs warnings and continues on file read failures

**Returns**: Dictionary mapping `clip_id → embedding_vector`. If insufficient embeddings are loaded, calling functions fall back to random selection.

#### `compute_medoids(...) -> List[str]`
Implements k-means clustering with medoid selection:

1. **Input validation**: Ensures sufficient clips for clustering
2. **Matrix construction**: Stacks embeddings into NumPy array
3. **K-means**: Uses scikit-learn with fixed seed for reproducibility
4. **Medoid extraction**: For each cluster, finds the clip closest to the centroid using L2 distance

**Medoids** (actual data points) are preferred over centroids (synthetic points) to ensure selected clips are representative and actually exist in the dataset.

#### `farthest_first_traversal(...) -> List[str]`
Implements greedy diversity maximization:

1. **Initialization**: Starts with specified seed or random point
2. **Distance tracking**: Maintains minimum distance from each point to the selected set
3. **Iterative selection**: Repeatedly chooses the farthest point, updating distances
4. **Termination**: Stops after selecting `n_select` clips

This algorithm ensures maximum spread in embedding space, useful for species enrichment where we want diverse examples per species.

---

### 4. Split Selection Functions

#### `select_spatial_test(...) -> pl.DataFrame`
**Spatial hold-out strategy**:

1. **Site grouping**: Uses `get_base_site()` to group recorder variants
2. **Random selection**: Shuffles base sites and selects first 5
3. **Month stratification**: Distributes 70 clips across available months for each site
4. **Constraint enforcement**: Calls `filter_with_constraints()` month-by-month
5. **Tracking**: Maintains global minute keys and spacing dictionaries across months

**Logs**: Reports site groupings, selection progress, and final distribution across actual site names.

#### `select_temporal_test(...) -> pl.DataFrame`
**Temporal hold-out strategy**:

1. **Site filtering**: Excludes spatial test sites to prevent overlap
2. **Edge day detection**: Identifies days 1-3 and last 3 days of each month using `monthrange()`
3. **Month iteration**: Selects 50 clips per month from edge days
4. **Constraint enforcement**: Uses shared minute key and spacing tracking

**Purpose**: Tests model robustness to temporal distribution shift, as edge-of-month conditions may differ from mid-month.

#### `select_diversity(...) -> pl.DataFrame`
**Diversity-based training selection**:

1. **Session tagging**: Adds "dawn" (hour < 12) vs "dusk" (hour ≥ 12) column
2. **Stratum identification**: Groups by (month, session) for 14 total strata
3. **Per-stratum selection**:
   - **Embedding path**: Attempts k-means medoid selection if embeddings available
   - **Fallback path**: Uses random selection if embeddings fail to load
   - **Supplementation**: Adds random clips if medoids don't satisfy constraints
4. **Site diversity**: Caps at 3 clips per site per stratum

**Rationale**: Medoids capture the "average" acoustic signature of each stratum, ensuring training coverage across temporal variation.

#### `select_species_enrichment(...) -> pl.DataFrame`
**Species-focused training augmentation**:

1. **Species ranking**: Identifies top 60 species by frequency in `top1_species` column
2. **Per-species processing**:
   - Filters to clips containing the species (from `top_species` list)
   - Extracts species score from parallel `top_scores` list
   - Sorts by score and takes top 2,000 candidates
3. **Diversity selection**:
   - **Embedding path**: Uses farthest-first traversal with 3× oversampling
   - **Fallback path**: Random selection from top candidates
4. **Site diversity**: Caps at 2 clips per site per species

**Rationale**: Ensures all common species have diverse training examples, preventing model bias toward a few species.

#### `create_dev_split(...) -> Tuple[Set[str], Set[str]]`
**Development set extraction**:

1. **File-level grouping**: Groups clips by (site, date, file_name)
2. **Month balancing**: Soft constraint to distribute dev clips across months (target: 43 clips/month)
3. **Group-wise selection**: Randomly selects entire file groups to prevent train/dev leakage within recordings
4. **Overshoot tolerance**: Allows up to 310 clips (instead of exactly 300) to avoid splitting files

**Returns**: Two sets of clip IDs (dev, remaining train)

---

### 5. Pipeline Integration

#### `add_split_columns(...) -> pl.DataFrame`
**Column generation**:

1. **Boolean flags**: Creates `is_spatial_test`, `is_temporal_test`, `is_train`, `is_dev`, etc.
2. **Eligibility**: Defines `eligible_train` (has embeddings, not in test sets)
3. **Pool identification**: Marks unused eligible clips for future selection
4. **Split label**: Categorical column with values: `train`, `dev`, `test_spatial`, `test_temporal`, `pool`, `excluded`

Uses Polars' expression API for efficient column-wise operations.

#### `validate_splits(...)`
**Sanity checks**:

1. **Count validation**: Logs actual vs. target counts for each split
2. **Overlap detection**: Asserts zero overlap between:
   - Spatial and temporal test sets
   - Test and train/dev sets
   - Dev and train sets
3. **Distribution logging**: Reports split label frequencies

Uses assertions that halt execution if critical invariants are violated.

---

### 6. Main Execution



In [ ]:
def main():
    # 1. Parse arguments (input/output paths, seed, --skip-embeddings flag)
    # 2. Initialize RNG with seed
    # 3. Load canonical index and log statistics
    # 4. Select spatial test (updates spatial_test_ids, spatial_test_sites)
    # 5. Select temporal test (excludes spatial sites)
    # 6. Filter to eligible training clips (not in test, has embeddings)
    # 7. Select diversity training (k-means or random)
    # 8. Select species enrichment (FFT or random)
    # 9. Create dev split from combined training
    # 10. Add all split columns to dataframe
    # 11. Validate splits (assertions + logging)
    # 12. Save annotated index to Parquet with Zstandard compression



**Command-line interface**:
- `--input`: Path to canonical clip index
- `--output`: Path for annotated output
- `--seed`: Random seed (default: 42)
- `--skip-embeddings`: Fast mode using only random selection

**Error handling**: Logs errors for empty selections but continues pipeline execution where possible, allowing partial results to be saved.

---

## Design Rationale

**Why medoids over random selection?**: Medoids capture representative examples of acoustic diversity rather than arbitrary samples, improving training efficiency.

**Why farthest-first for species?**: Ensures each species is represented by maximally diverse examples, preventing overfitting to specific call variants.

**Why group-based dev split?**: Prevents train/dev leakage from temporally adjacent clips in the same recording file.

**Why spatial AND temporal test sets?**: Tests two orthogonal distribution shifts—generalization to new locations and new time periods—critical for real-world deployment.